## Scaling — changing the desired count

Scaling a Deployment is a one-field change; the controller does the rest.

```bash
kubectl scale deployment web --replicas=5   # imperative, fastest for the exam
kubectl edit deployment web                 # or edit spec.replicas + kubectl apply -f
```

When `replicas` changes:

- **Scale up.** The Deployment updates the active ReplicaSet's `replicas`; the ReplicaSet controller sees the shortfall and creates the difference. Each new pod walks `Pending` → `ContainerCreating` → `Running` → `Ready`.
- **Scale down.** The ReplicaSet controller deletes the excess, preferring `NotReady` pods, then pods on nodes already carrying more of the set, then by age. Chosen pods get `SIGTERM`, their `terminationGracePeriodSeconds`, then `SIGKILL`.

Crucially, **scaling does not create a new ReplicaSet.** Same template, same generation, just a different count — so it is *not* a rollout and won't appear in `kubectl rollout history`. That distinction (scale vs rollout) is a common exam trip-up.

### HorizontalPodAutoscaler — a glance

`replicas` is the manual lever. An **HPA** delegates it to a metric:

```bash
kubectl autoscale deployment web --min=2 --max=10 --cpu-percent=70
```

The HPA controller polls the metric (~every 15s) and adjusts `replicas` to keep CPU near 70%. Full autoscaling comes with resources in notebook 07. On our map, scaling is the **scale** verb on the left driving the **ReplicaSet** count — a change in *how many* Pods, never *what* they run.